In [3]:
print("Hellosrdrghtew")

Hellosrdrghtew


In [2]:
print("yaswanth alam")

yaswanth alam


In [1]:
print("testing")

testing


Write a Python function that takes a string and checks if it is a valid IPv4 address.

In [6]:
def is_valid_ipv4(ip_str):
    parts = ip_str.split(".")

    if len(parts) != 4:
        return False
    for part in parts:
        if not part.isdigit():
            return False
        
        if len(part) > 1 and part.startswith("0"):
            return False
        
        value = int(part)
        if value < 0 or value > 255:
            return False
        
    return True

In [7]:
print(is_valid_ipv4("192.168.1.1"))     

True


In [8]:
print(is_valid_ipv4("256.1.1.1"))

False


In [9]:
print(is_valid_ipv4("192.168.01.1"))

False


Subnet Parsing

Given a list of CIDRs, print:

Network address
Broadcast address
Number of usable hosts

In [11]:
!pip install ipaddress

In [13]:
import ipaddress

def print_subnet_details(cidrs):
    for cidr in cidrs:
        try:
            # strict=False allows inputs like "192.168.1.5/24" and treats it as "192.168.1.0/24"
            net = ipaddress.ip_network(cidr, strict=False)

            network_addr = net.network_address

            # Broadcast exists only for IPv4 networks (ip_network returns correct object for v4/v6)
            broadcast_addr = net.broadcast_address if isinstance(net, ipaddress.IPv4Network) else None

            total_ips = net.num_addresses

            # Usable hosts logic (IPv4):
            # /32 -> 1 address (host itself)
            # /31 -> 2 addresses (RFC 3021 point-to-point), both usable
            # other -> total - 2 (exclude network + broadcast)
            if isinstance(net, ipaddress.IPv4Network):
                if net.prefixlen == 32:
                    usable_hosts = 1
                elif net.prefixlen == 31:
                    usable_hosts = 2
                else:
                    usable_hosts = max(total_ips - 2, 0)
            else:
                # IPv6: no broadcast concept, "usable" depends on your definition
                usable_hosts = total_ips

            print(f"CIDR: {cidr}")
            print(f"  Network Address   : {network_addr}")
            if broadcast_addr is not None:
                print(f"  Broadcast Address : {broadcast_addr}")
            else:
                print(f"  Broadcast Address : N/A (IPv6)")
            print(f"  Usable Hosts      : {usable_hosts}")
            print("-" * 50)

        except ValueError as e:
            print(f"CIDR: {cidr} -> Invalid CIDR ({e})")
            print("-" * 50)


# Example usage
cidrs = [
    "192.168.1.0/24",
    "10.0.0.0/30",
    "10.0.0.0/31",
    "10.0.0.1/32",
    "192.168.1.5/24",   # valid because strict=False normalizes it
    "300.1.1.0/24"      # invalid example
]

print_subnet_details(cidrs)


CIDR: 192.168.1.0/24
  Network Address   : 192.168.1.0
  Broadcast Address : 192.168.1.255
  Usable Hosts      : 254
--------------------------------------------------
CIDR: 10.0.0.0/30
  Network Address   : 10.0.0.0
  Broadcast Address : 10.0.0.3
  Usable Hosts      : 2
--------------------------------------------------
CIDR: 10.0.0.0/31
  Network Address   : 10.0.0.0
  Broadcast Address : 10.0.0.1
  Usable Hosts      : 2
--------------------------------------------------
CIDR: 10.0.0.1/32
  Network Address   : 10.0.0.1
  Broadcast Address : 10.0.0.1
  Usable Hosts      : 1
--------------------------------------------------
CIDR: 192.168.1.5/24
  Network Address   : 192.168.1.0
  Broadcast Address : 192.168.1.255
  Usable Hosts      : 254
--------------------------------------------------
CIDR: 300.1.1.0/24 -> Invalid CIDR ('300.1.1.0/24' does not appear to be an IPv4 or IPv6 network)
--------------------------------------------------


Config Line Counter

Given a router configuration file, count:

Number of interfaces
Number of shutdown interfaces

In [14]:
def count_interfaces_and_shutdowns(config_path):
    interface_count = 0
    shutdown_interface_count = 0

    current_iface = None          # stores current interface name
    current_iface_is_shutdown = False

    with open(config_path, "r", encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.strip()

            # Skip empty lines
            if not line:
                continue

            # Detect start of an interface block
            if line.startswith("interface "):
                # If we were already inside an interface, finalize it
                if current_iface is not None:
                    interface_count += 1
                    if current_iface_is_shutdown:
                        shutdown_interface_count += 1

                # Start tracking the new interface
                current_iface = line.split(None, 1)[1]   # everything after 'interface '
                current_iface_is_shutdown = False
                continue

            # Detect end of a block (common delimiter in Cisco configs)
            if line == "!":
                if current_iface is not None:
                    interface_count += 1
                    if current_iface_is_shutdown:
                        shutdown_interface_count += 1

                    # Reset since interface block ended
                    current_iface = None
                    current_iface_is_shutdown = False
                continue

            # If we're inside an interface block, look for shutdown
            if current_iface is not None:
                if line == "shutdown":
                    current_iface_is_shutdown = True

    # End of file: finalize last interface block if config doesn't end with "!"
    if current_iface is not None:
        interface_count += 1
        if current_iface_is_shutdown:
            shutdown_interface_count += 1

    return interface_count, shutdown_interface_count


# ---------------- Example usage ----------------
if __name__ == "__main__":
    config_file = "router.cfg"  # change this to your file path
    interfaces, shutdowns = count_interfaces_and_shutdowns(config_file)

    print(f"Total interfaces         : {interfaces}")
    print(f"Shutdown interfaces      : {shutdowns}")

FileNotFoundError: [Errno 2] No such file or directory: 'router.cfg'

In [15]:
import ipaddress

def subnet_info(cidr: str):
    network = ipaddress.IPv4Network(cidr, strict=False)
    print(f"Network Address  : {network.network_address}")
    print(f"Broadcast Address: {network.broadcast_address}")
    print(f"Subnet Mask      : {network.netmask}")
    print(f"Usable Hosts     : {network.num_addresses - 2}")
    print(f"Host Range       : {list(network.hosts())[0]} - {list(network.hosts())[-1]}")

subnet_info("192.168.1.0/24")

Network Address  : 192.168.1.0
Broadcast Address: 192.168.1.255
Subnet Mask      : 255.255.255.0
Usable Hosts     : 254
Host Range       : 192.168.1.1 - 192.168.1.254


In [16]:
import ipaddress
import subprocess
import platform

def ping_host(ip: str) -> bool:
    param = "-n" if platform.system().lower() == "windows" else "-c"
    result = subprocess.run(
        ["ping", param, "1", "-W", "1", str(ip)],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )
    return result.returncode == 0

def ping_sweep(subnet: str):
    network = ipaddress.IPv4Network(subnet, strict=False)
    for host in network.hosts():
        status = "UP  ✅" if ping_host(str(host)) else "DOWN ❌"
        print(f"{host:20} {status}")

ping_sweep("192.168.1.0/28")  # Use a small subnet for testing

FileNotFoundError: [Errno 2] No such file or directory: 'ping'

In [17]:
import re

route_table = """
O    10.0.0.0/8 [110/20] via 192.168.1.1, 00:01:00, GigabitEthernet0/0
C    192.168.1.0/24 is directly connected, GigabitEthernet0/0
S    172.16.0.0/16 [1/0] via 10.1.1.1
O    10.1.1.0/24 [110/30] via 192.168.1.2, 00:00:45, GigabitEthernet0/1
"""

def parse_routing_table(table: str) -> list:
    routes = []
    pattern = re.compile(
        r'(\w+)\s+([\d./]+).*?via\s+([\d.]+)(?:.*?(GigabitEthernet[\w/]+|FastEthernet[\w/]+))?'
    )
    for line in table.strip().splitlines():
        match = pattern.search(line)
        if match:
            routes.append({
                "type"     : match.group(1),
                "network"  : match.group(2),
                "next_hop" : match.group(3),
                "interface": match.group(4) or "N/A"
            })
    return routes

for route in parse_routing_table(route_table):
    print(route)

{'type': 'O', 'network': '10.0.0.0/8', 'next_hop': '192.168.1.1', 'interface': 'GigabitEthernet0/0'}
{'type': 'S', 'network': '172.16.0.0/16', 'next_hop': '10.1.1.1', 'interface': 'N/A'}
{'type': 'O', 'network': '10.1.1.0/24', 'next_hop': '192.168.1.2', 'interface': 'GigabitEthernet0/1'}


In [18]:
import socket

def port_scanner(host: str, ports, timeout: float = 0.5) -> list:
    open_ports = []
    for port in ports:
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(timeout)
                result = s.connect_ex((host, port))
                if result == 0:
                    open_ports.append(port)
                    print(f"  Port {port:5} OPEN")
        except socket.error:
            pass
    return open_ports

print("Scanning...")
results = port_scanner("scanme.nmap.org", range(20, 100))
print(f"\nOpen ports: {results}")

Scanning...
  Port    22 OPEN
  Port    80 OPEN

Open ports: [22, 80]


In [20]:
import socket

def port_scanner(host: str, ports, timeout: float = 0.5) -> list:
    open_ports = []
    for port in ports:
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(timeout)
                result = s.connect_ex((host, port))
                if result == 0:
                    open_ports.append(port)
                    print(f"  Port {port:5} OPEN")
        except socket.error:
            pass
    return open_ports

print("Scanning...")
results = port_scanner("www.amazon.in", range(20, 100))
print(f"\nOpen ports: {results}")

Scanning...
  Port    80 OPEN

Open ports: [80]


In [ ]:
import socket

def port_scanner(host: str, ports, timeout: float = 0.5) -> list:
    open_ports = []
    for port in ports:
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(timeout)
                result = s.connect_ex((host, port))
                if result == 0:
                    open_ports.append(port)
                    print(f"  Port {port:5} OPEN")
        except socket.error:
            pass
    return open_ports

print("Scanning...")
results = port_scanner("amazon.in", range(20, 1000))
print(f"\nOpen ports: {results}")

Scanning...
  Port    80 OPEN
  Port   443 OPEN


In [1]:
import socket

def port_scanner(host: str, ports, timeout: float = 0.5) -> list:
    open_ports = []
    for port in ports:
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(timeout)
                result = s.connect_ex((host, port))
                if result == 0:
                    open_ports.append(port)
                    print(f"  Port {port:5} OPEN")
        except socket.error:
            pass
    return open_ports

print("Scanning...")
results = port_scanner("openai.com", range(20, 1000))
print(f"\nOpen ports: {results}")

Scanning...
  Port    80 OPEN
  Port   443 OPEN

Open ports: [80, 443]


In [ ]:
import socket

def port_scanner(host: str, ports, timeout: float = 0.5) -> list:
    open_ports = []
    for port in ports:
        try:
            with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
                s.settimeout(timeout)
                result = s.connect_ex((host, port))
                if result == 0:
                    open_ports.append(port)
                    print(f"  Port {port:5} OPEN")
        except socket.error:
            pass
    return open_ports

print("Scanning...")
results = port_scanner("chatgpt.com", range(20, 1000))
print(f"\nOpen ports: {results}")

Scanning...
  Port    80 OPEN
  Port   443 OPEN


In [1]:
import re

vlan_output = """
VLAN Name                             Status    Ports
---- -------------------------------- --------- -------------------------------
1    default                          active    Gi0/1, Gi0/2
10   MANAGEMENT                       active    Gi0/3, Gi0/4, Gi0/5
20   SERVERS                          active    Gi1/0, Gi1/1
99   NATIVE                           active    
"""

def parse_vlan_table(output: str) -> dict:
    vlans = {}
    pattern = re.compile(r'^(\d+)\s+(\S+)\s+\S+\s*(.*)?$')
    for line in output.strip().splitlines():
        match = pattern.match(line.strip())
        if match:
            vlan_id = match.group(1)
            name    = match.group(2)
            ports   = [p.strip() for p in match.group(3).split(",") if p.strip()]
            vlans[vlan_id] = {"name": name, "ports": ports}
    return vlans

vlans = parse_vlan_table(vlan_output)
for vid, info in vlans.items():
    print(f"VLAN {vid}: {info}")

VLAN 1: {'name': 'default', 'ports': ['Gi0/1', 'Gi0/2']}
VLAN 10: {'name': 'MANAGEMENT', 'ports': ['Gi0/3', 'Gi0/4', 'Gi0/5']}
VLAN 20: {'name': 'SERVERS', 'ports': ['Gi1/0', 'Gi1/1']}
VLAN 99: {'name': 'NATIVE', 'ports': []}


In [2]:
import re
from collections import defaultdict

bgp_log = """
Mar 10 10:01:00 router bgpd: %BGP-5-ADJCHANGE: neighbor 10.0.0.1 Up
Mar 10 10:05:00 router bgpd: %BGP-5-ADJCHANGE: neighbor 10.0.0.1 Down
Mar 10 10:10:00 router bgpd: %BGP-5-ADJCHANGE: neighbor 10.0.0.2 Up
Mar 10 10:15:00 router bgpd: %BGP-5-ADJCHANGE: neighbor 10.0.0.1 Up
Mar 10 10:20:00 router bgpd: %BGP-5-ADJCHANGE: neighbor 10.0.0.2 Down
Mar 10 10:25:00 router bgpd: %BGP-5-ADJCHANGE: neighbor 10.0.0.1 Down
"""

def analyze_bgp_log(log: str):
    pattern = re.compile(r'(\w+ \d+ \d+:\d+:\d+).*neighbor ([\d.]+) (Up|Down)')
    stats = defaultdict(lambda: {"Up": 0, "Down": 0, "events": []})

    for line in log.strip().splitlines():
        match = pattern.search(line)
        if match:
            timestamp, neighbor, state = match.groups()
            stats[neighbor][state] += 1
            stats[neighbor]["events"].append((timestamp, state))

    for neighbor, data in stats.items():
        print(f"\nNeighbor: {neighbor}")
        print(f"  Up Count  : {data['Up']}")
        print(f"  Down Count: {data['Down']}")
        print(f"  Events:")
        for ts, state in data["events"]:
            print(f"    [{ts}] -> {state}")

analyze_bgp_log(bgp_log)


Neighbor: 10.0.0.1
  Up Count  : 2
  Down Count: 2
  Events:
    [Mar 10 10:01:00] -> Up
    [Mar 10 10:05:00] -> Down
    [Mar 10 10:15:00] -> Up
    [Mar 10 10:25:00] -> Down

Neighbor: 10.0.0.2
  Up Count  : 1
  Down Count: 1
  Events:
    [Mar 10 10:10:00] -> Up
    [Mar 10 10:20:00] -> Down


In [3]:
!pip install pysnmp

Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 0.0/6.0 MB ? eta -:--:--
     ---------------------------------------- 6.0/6.0 MB 40.1 MB/s  0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for pysmp: filename=pysmp-0.1.3-py3-none-any.whl size=6049585 sha256=ec0d59e8cf89f61cc27cce2c257791f783e55cd6eb425ec94cc71c58f7dd5dfa
  Stored in directory: c:\users\v-yaalam\appdata\local\packages\pythonsoftwarefoundation.python.3.13_qbz5n2kfra8p0\localcache\local\pip\cache\wheels\33\ad\1b\fd863d7c83020aa48380a3d9a552bafe339cda8333d3cb7d9c
Successfully built pysmp



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\v-yaalam\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
# pip install pysnmp
import time
from pysnmp.hlapi import *

def get_snmp_counter(host, community, oid):
    iterator = getCmd(
        SnmpEngine(),
        CommunityData(community),
        UdpTransportTarget((host, 161), timeout=2, retries=1),
        ContextData(),
        ObjectType(ObjectIdentity(oid))
    )
    errorIndication, errorStatus, errorIndex, varBinds = next(iterator)
    if errorIndication or errorStatus:
        return None
    return int(varBinds[0][1])

def poll_bandwidth(host: str, community: str, interface_index: int, interval: int = 30):
    in_oid  = f"1.3.6.1.2.1.2.2.1.10.{interface_index}"
    out_oid = f"1.3.6.1.2.1.2.2.1.16.{interface_index}"

    prev_in = get_snmp_counter(host, community, in_oid)
    prev_out = get_snmp_counter(host, community, out_oid)
    print(f"Polling {host} every {interval}s...\n")

    while True:
        time.sleep(interval)
        curr_in  = get_snmp_counter(host, community, in_oid)
        curr_out = get_snmp_counter(host, community, out_oid)

        if curr_in and curr_out:
            bps_in  = ((curr_in  - prev_in)  * 8) / interval
            bps_out = ((curr_out - prev_out) * 8) / interval
            print(f"  IN : {bps_in/1000:.2f} Kbps")
            print(f"  OUT: {bps_out/1000:.2f} Kbps\n")
            prev_in, prev_out = curr_in, curr_out

poll_bandwidth("192.168.1.1", "public", interface_index=1)

ModuleNotFoundError: No module named 'pysnmp'

In [1]:
n = 5

for i in range(1, n+1):
    print("*"*i)

*
**
***
****
*****
